# Conformal label-set prediction

Calibrate label sets on matched internal objects and evaluate coverage on separate images. Coverage is measured among spatially matched objects.

## Load the data

Place the downloaded files in `data/`, or set `GFLD_DATA_DIR` to their location. Dataset details are provided in the paper.

In [ ]:
%matplotlib inline
from pathlib import Path
from dataclasses import asdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from gfld import *

# Works when Jupyter starts in the repository root or notebooks directory.
ROOT = Path.cwd() if (Path.cwd() / "pyproject.toml").is_file() else Path.cwd().parent
DATA_DIR = example_directory(ROOT)  # GFLD_DATA_DIR or repository data/
data = load_example(DATA_DIR)
print({k: v["images"] for k, v in data["metadata"]["splits"].items()})

In [ ]:
# Split calibration and evaluation by image.
_, calibration_idx, evaluation_idx = next(calibration_splits(
    len(data["internal"]), folds=5, permutations=1, seed=42))
calibration = [data["internal"][i] for i in calibration_idx]
evaluation = [data["internal"][i] for i in evaluation_idx]
annotations = data["internal_annotations"]
assert set(p.image_id for p in calibration).isdisjoint(p.image_id for p in evaluation)
print(f"{len(calibration)} calibration images; {len(evaluation)} held-out images")

## Calibrate the class-score threshold

Collect each matched object’s true-class score and compute the calibration quantile.

In [ ]:
cp = calibrate_cp(calibration, annotations, alpha=0.01)
pd.DataFrame([asdict(cp)])

In [ ]:
from gfld.conformal import collect_scores
scores, matched_image_ids = collect_scores(calibration, annotations)
plt.hist(scores, bins=35)
plt.axvline(cp.threshold, color="tab:red", label=f"threshold = {cp.threshold:.4f}")
plt.xlabel("Score assigned to the matched true class"); plt.ylabel("Matched calibration pairs")
plt.legend(); plt.show()

## Predict label sets

Include every class whose score meets the calibrated threshold.

In [ ]:
p = evaluation[0]
sets = cp.predict(p.class_scores)
pd.DataFrame({"confidence": p.scores,
              "label_set": [[CLASSES[c] for c in np.flatnonzero(row)] for row in sets]})

In [ ]:
pd.DataFrame([evaluate_cp(evaluation, annotations, cp)])

## Compare coverage levels

Evaluate coverage and average set size for several miscoverage levels.

In [ ]:
rows = []
for alpha in (0.10, 0.05, 0.01):
    fitted = calibrate_cp(calibration, annotations, alpha=alpha)
    rows.append({"nominal_coverage": 1-alpha, "threshold": fitted.threshold,
                 **evaluate_cp(evaluation, annotations, fitted)})
pd.DataFrame(rows)